<h1> Author: Soham Shah </h1>

Q1: Run K-means clustering with Euclidean, Cosine and Jarcard similarity. Specify K= the
number of categorical values of y (the number of classifications). Compare the SSEs of
Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which method is better? (10 points)

In [1]:
import numpy as np
import pandas as pd

data = pd.read_csv('data.csv', header=None).values
labels = pd.read_csv('label.csv', header=None).values.flatten()
k = len(np.unique(labels))

def kmeans(X, k, metric, max_iters=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    
    for _ in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[labels == i].mean(axis=0) for i in range(k)])
        
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    
    sse = sum(np.sum(distances[labels == i, i] ** 2) for i in range(k))
    return sse

sse_euclidean = kmeans(data, k, 'euclidean')
sse_cosine = kmeans(data, k, 'cosine')
sse_jaccard = kmeans(data, k, 'jaccard')

print(f"Euclidean SSE: {sse_euclidean:.2f}")
print(f"Cosine SSE: {sse_cosine:.2f}")
print(f"Jaccard SSE: {sse_jaccard:.2f}")

best = min([('Euclidean', sse_euclidean), ('Cosine', sse_cosine), ('Jaccard', sse_jaccard)], key=lambda x: x[1])
print(f"\nBest method: {best[0]}")


Euclidean SSE: 25414767689.96
Cosine SSE: 686.44
Jaccard SSE: 3660.39

Best method: Cosine


Q2: Compare the accuracies of Euclidean-K-means Cosine-K-means, Jarcard-K-means. First,
label each cluster using the majority vote label of the data points in that cluster. Later, compute
the predictive accuracy of Euclidean-K-means, Cosine-K-means, Jarcard-K-means. Which metric
is better? (10 points)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import mode

data = pd.read_csv('data.csv', header=None).values
true_labels = pd.read_csv('label.csv', header=None).values.flatten()
k = len(np.unique(true_labels))

def kmeans(X, k, metric, max_iters=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    
    for _ in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        
        if np.allclose(centroids, new_centroids):
            break
        centroids = new_centroids
    
    return cluster_labels

def compute_accuracy(cluster_labels, true_labels):
    predicted_labels = np.zeros_like(cluster_labels)
    
    for i in range(k):
        mask = cluster_labels == i
        if np.sum(mask) > 0:
            majority_label = mode(true_labels[mask], keepdims=True)[0][0]
            predicted_labels[mask] = majority_label
    
    accuracy = np.sum(predicted_labels == true_labels) / len(true_labels)
    return accuracy

cluster_euclidean = kmeans(data, k, 'euclidean')
cluster_cosine = kmeans(data, k, 'cosine')
cluster_jaccard = kmeans(data, k, 'jaccard')

acc_euclidean = compute_accuracy(cluster_euclidean, true_labels)
acc_cosine = compute_accuracy(cluster_cosine, true_labels)
acc_jaccard = compute_accuracy(cluster_jaccard, true_labels)

print(f"Euclidean Accuracy: {acc_euclidean:.4f}")
print(f"Cosine Accuracy: {acc_cosine:.4f}")
print(f"Jaccard Accuracy: {acc_jaccard:.4f}")

best = max([('Euclidean', acc_euclidean), ('Cosine', acc_cosine), ('Jaccard', acc_jaccard)], key=lambda x: x[1])
print(f"\nBest method: {best[0]}")

Euclidean Accuracy: 0.5851
Cosine Accuracy: 0.6309
Jaccard Accuracy: 0.6021

Best method: Cosine


Q3: Set up the same stop criteria: “when there is no change in centroid position OR when the
SSE value increases in the next iteration OR when the maximum preset value (e.g., 500, you
can set the preset value by yourself) of iteration is complete”, for Euclidean-K-means, Cosine-Kmeans,
Jarcard-K-means. Which method requires more iterations and times to converge? (10
points)

In [ ]:
import numpy as np
import pandas as pd
import time

data = pd.read_csv('data.csv', header=None).values
labels = pd.read_csv('label.csv', header=None).values.flatten()
k = len(np.unique(labels))

def kmeans_case1(X, k, metric, max_iters=500):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    iterations = 0
    start_time = time.time()
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        
        iterations = iteration + 1
        
        if np.allclose(centroids, new_centroids):
            break
        
        centroids = new_centroids
    
    elapsed_time = time.time() - start_time
    return iterations, elapsed_time

def kmeans_case2(X, k, metric, max_iters=500):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    prev_sse = float('inf')
    iterations = 0
    start_time = time.time()
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        current_sse = sum(np.sum(distances[cluster_labels == i, i] ** 2) for i in range(k))
        
        iterations = iteration + 1
        
        if current_sse > prev_sse:
            break
        
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        centroids = new_centroids
        prev_sse = current_sse
    
    elapsed_time = time.time() - start_time
    return iterations, elapsed_time

def kmeans_case3(X, k, metric, max_iters=500):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    iterations = 0
    start_time = time.time()
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        
        iterations = iteration + 1
        
        if iteration >= max_iters - 1:
            break
        
        centroids = new_centroids
    
    elapsed_time = time.time() - start_time
    return iterations, elapsed_time

print("Case 1: No change in centroid position")
print("="*50)
iters_e1, time_e1 = kmeans_case1(data, k, 'euclidean')
iters_c1, time_c1 = kmeans_case1(data, k, 'cosine')
iters_j1, time_j1 = kmeans_case1(data, k, 'jaccard')
print(f"Euclidean - Iterations: {iters_e1}, Time: {time_e1:.4f}s")
print(f"Cosine - Iterations: {iters_c1}, Time: {time_c1:.4f}s")
print(f"Jaccard - Iterations: {iters_j1}, Time: {time_j1:.4f}s")

print("\nCase 2: SSE increases in next iteration")
print("="*50)
iters_e2, time_e2 = kmeans_case2(data, k, 'euclidean')
iters_c2, time_c2 = kmeans_case2(data, k, 'cosine')
iters_j2, time_j2 = kmeans_case2(data, k, 'jaccard')
print(f"Euclidean - Iterations: {iters_e2}, Time: {time_e2:.4f}s")
print(f"Cosine - Iterations: {iters_c2}, Time: {time_c2:.4f}s")
print(f"Jaccard - Iterations: {iters_j2}, Time: {time_j2:.4f}s")

print("\nCase 3: Maximum preset value (500 iterations)")
print("="*50)
iters_e3, time_e3 = kmeans_case3(data, k, 'euclidean', max_iters=500)
iters_c3, time_c3 = kmeans_case3(data, k, 'cosine', max_iters=500)
iters_j3, time_j3 = kmeans_case3(data, k, 'jaccard', max_iters=500)
print(f"Euclidean - Iterations: {iters_e3}, Time: {time_e3:.4f}s")
print(f"Cosine - Iterations: {iters_c3}, Time: {time_c3:.4f}s")
print(f"Jaccard - Iterations: {iters_j3}, Time: {time_j3:.4f}s")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)

all_results = [
    ('Euclidean-Case1', iters_e1, time_e1),
    ('Cosine-Case1', iters_c1, time_c1),
    ('Jaccard-Case1', iters_j1, time_j1),
    ('Euclidean-Case2', iters_e2, time_e2),
    ('Cosine-Case2', iters_c2, time_c2),
    ('Jaccard-Case2', iters_j2, time_j2),
    ('Euclidean-Case3', iters_e3, time_e3),
    ('Cosine-Case3', iters_c3, time_c3),
    ('Jaccard-Case3', iters_j3, time_j3)
]

most_iters = max(all_results, key=lambda x: x[1])
most_time = max(all_results, key=lambda x: x[2])

print(f"Most iterations: {most_iters[0]} with {most_iters[1]} iterations")
print(f"Most time: {most_time[0]} with {most_time[2]:.4f}s")

Case 1: No change in centroid position
Euclidean - Iterations: 33, Time: 15.2675s
Cosine - Iterations: 48, Time: 26.8157s
Jaccard - Iterations: 59, Time: 29.9563s

Case 2: SSE increases in next iteration
Euclidean - Iterations: 500, Time: 227.9082s
Cosine - Iterations: 29, Time: 17.2798s
Jaccard - Iterations: 2, Time: 0.8689s

Case 3: Maximum preset value (500 iterations)
Euclidean - Iterations: 500, Time: 233.0024s
Cosine - Iterations: 500, Time: 307.6632s
Jaccard - Iterations: 500, Time: 288.3799s

SUMMARY
Most iterations: Euclidean-Case2 with 500 iterations
Most time: Cosine-Case3 with 307.6632s


Q4: Compare the SSEs of Euclidean-K-means Cosine-K-means, Jarcard-K-means with respect to
the following three terminating conditions: (10 points)
- when there is no change in centroid position
- when the SSE value increases in the next iteration
- when the maximum preset value (e.g., 100) of iteration is complete

In [ ]:
import numpy as np
import pandas as pd

data = pd.read_csv('data.csv', header=None).values
labels = pd.read_csv('label.csv', header=None).values.flatten()
k = len(np.unique(labels))

def kmeans_case1(X, k, metric, max_iters=500):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        
        if np.allclose(centroids, new_centroids):
            break
        
        centroids = new_centroids
    
    sse = sum(np.sum(distances[cluster_labels == i, i] ** 2) for i in range(k))
    return sse

def kmeans_case2(X, k, metric, max_iters=500):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    prev_sse = float('inf')
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        current_sse = sum(np.sum(distances[cluster_labels == i, i] ** 2) for i in range(k))
        
        if current_sse > prev_sse:
            break
        
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        centroids = new_centroids
        prev_sse = current_sse
    
    return prev_sse

def kmeans_case3(X, k, metric, max_iters=100):
    np.random.seed(42)
    centroids = X[np.random.choice(X.shape[0], k, replace=False)]
    
    for iteration in range(max_iters):
        distances = np.zeros((X.shape[0], k))
        
        for i in range(k):
            if metric == 'euclidean':
                distances[:, i] = np.sqrt(np.sum((X - centroids[i]) ** 2, axis=1))
            elif metric == 'cosine':
                dot = np.dot(X, centroids[i])
                norm = np.linalg.norm(X, axis=1) * np.linalg.norm(centroids[i])
                distances[:, i] = 1 - dot / (norm + 1e-10)
            elif metric == 'jaccard':
                min_sum = np.sum(np.minimum(X, centroids[i]), axis=1)
                max_sum = np.sum(np.maximum(X, centroids[i]), axis=1)
                distances[:, i] = 1 - min_sum / (max_sum + 1e-10)
        
        cluster_labels = np.argmin(distances, axis=1)
        new_centroids = np.array([X[cluster_labels == i].mean(axis=0) for i in range(k)])
        centroids = new_centroids
    
    sse = sum(np.sum(distances[cluster_labels == i, i] ** 2) for i in range(k))
    return sse

print("Case 1: No change in centroid position")
print("="*50)
sse_e1 = kmeans_case1(data, k, 'euclidean')
sse_c1 = kmeans_case1(data, k, 'cosine')
sse_j1 = kmeans_case1(data, k, 'jaccard')
print(f"Euclidean SSE: {sse_e1:.2f}")
print(f"Cosine SSE: {sse_c1:.2f}")
print(f"Jaccard SSE: {sse_j1:.2f}")
best1 = min([('Euclidean', sse_e1), ('Cosine', sse_c1), ('Jaccard', sse_j1)], key=lambda x: x[1])
print(f"Best: {best1[0]}")

print("\nCase 2: SSE increases in next iteration")
print("="*50)
sse_e2 = kmeans_case2(data, k, 'euclidean')
sse_c2 = kmeans_case2(data, k, 'cosine')
sse_j2 = kmeans_case2(data, k, 'jaccard')
print(f"Euclidean SSE: {sse_e2:.2f}")
print(f"Cosine SSE: {sse_c2:.2f}")
print(f"Jaccard SSE: {sse_j2:.2f}")
best2 = min([('Euclidean', sse_e2), ('Cosine', sse_c2), ('Jaccard', sse_j2)], key=lambda x: x[1])
print(f"Best: {best2[0]}")

print("\nCase 3: Maximum preset value (100 iterations)")
print("="*50)
sse_e3 = kmeans_case3(data, k, 'euclidean', max_iters=100)
sse_c3 = kmeans_case3(data, k, 'cosine', max_iters=100)
sse_j3 = kmeans_case3(data, k, 'jaccard', max_iters=100)
print(f"Euclidean SSE: {sse_e3:.2f}")
print(f"Cosine SSE: {sse_c3:.2f}")
print(f"Jaccard SSE: {sse_j3:.2f}")
best3 = min([('Euclidean', sse_e3), ('Cosine', sse_c3), ('Jaccard', sse_j3)], key=lambda x: x[1])
print(f"Best: {best3[0]}")

print("\n" + "="*50)
print("SUMMARY")
print("="*50)
print(f"Case 1 Best: {best1[0]} (SSE: {best1[1]:.2f})")
print(f"Case 2 Best: {best2[0]} (SSE: {best2[1]:.2f})")
print(f"Case 3 Best: {best3[0]} (SSE: {best3[1]:.2f})")

Case 1: No change in centroid position
Euclidean SSE: 25414767689.96
Cosine SSE: 686.44
Jaccard SSE: 3660.39
Best: Cosine

Case 2: SSE increases in next iteration
Euclidean SSE: 25414767689.96
Cosine SSE: 686.19
Jaccard SSE: 4196.27
Best: Cosine

Case 3: Maximum preset value (100 iterations)
Euclidean SSE: 25414767689.96
Cosine SSE: 686.44
Jaccard SSE: 3660.39
Best: Cosine

SUMMARY
Case 1 Best: Cosine (SSE: 686.44)
Case 2 Best: Cosine (SSE: 686.19)
Case 3 Best: Cosine (SSE: 686.44)
